In [1]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

Error importing in API mode: ImportError("dlopen(/Users/nataliejonas/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <B96A8100-FA7A-3EFC-8726-931D26646DE6> /Users/nataliejonas/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


In [2]:
%%javascript
// Disable auto-scrolling
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [3]:
%%R

require('tidyverse')
require('broom')
require(readr)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: tidyverse
Loading required package: broom


In [4]:
%%R
 
wage <- read_csv('wage_theft.csv', show_col_types = FALSE)
wage

# A tibble: 3,212 × 10
   `Year of Case` `Employer Name`          Address Street City  State `Zip Code`
            <dbl> <chr>                    <chr>   <chr>  <chr> <chr>      <dbl>
 1           2020 ABI                      131-07… 131-0… Flus… NY         11354
 2           2022 Royal Care               6323 1… 6323 … Broo… NY         11219
 3           2022 American Business Insti… 131-07… 131-0… Flus… NY         11354
 4           2020 Ultimate Services For Y… 1506 S… 1506 … Broo… NY         11235
 5           2022 Renaissance Home Health… 267 Do… 267 D… Broo… NY         11217
 6           2020 American Chore Services… 2719 C… 2719 … Broo… NY         11235
 7           2021 Longevity Health Servic… 142-29… 142-2… Flus… NY         11354
 8           2022 Allstar Home Care Agenc… 762 59… 762 5… Broo… NY         11219
 9           2021 Nocello Ristorante Ital… 14-27 … 14-27… Whit… NY         11357
10           2022 ABI/American Business I… 131-07… 131-0… Flus… NY         11354
# ℹ 3

In [5]:
# data in python
wage = pd.read_csv('wage_theft.csv')
wage.head()

,Year of Case,Employer Name,Address,Street,City,State,Zip Code,Wages Owed,Number of Claimants on Case,USDOL?
0,2020,ABI,"131-07 40th Road, Flushing, NY, 11354",131-07 40th Road,Flushing,NY,11354,"$682,581.00",NaN,NO
1,2022,Royal Care,"6323 14th Ave., Brooklyn, NY, 11219",6323 14th Ave.,Brooklyn,NY,11219,"$5,851,021.00",108.0,NO
2,2022,American Business Institute,"131-07 40th Rd, Flushing, NY, 11354",131-07 40th Rd,Flushing,NY,11354,"$6,270,141.00",NaN,NO
3,2020,Ultimate Services For You Inc,"1506 Sheepshead Bay Road, Brooklyn, NY, 11235",1506 Sheepshead Bay Road,Brooklyn,NY,11235,"$2,223,856.00",265.0,NO
4,2022,Renaissance Home Health Care Services,"267 Douglass Street, Brooklyn, NY, 11217",267 Douglass Street,Brooklyn,NY,11217,"$1,781,707.00",14.0,NO


In [6]:
%%R

wage_clean1 <- wage %>%
  mutate(
    wages_owed = parse_number(`Wages Owed`),
    usdol = as.integer(`USDOL?` == "YES"),
    year = `Year of Case`,
    claimants = `Number of Claimants on Case`
  ) %>%
  filter(!is.na(wages_owed), !is.na(claimants))

# claimants: more workers on a case means more total wages owed
# year: enforcement patterns and case sizes can vary over time
# usdol: federal DOL involvement can signal more serious or larger cases
# city: labor and enforcement also varies according to geography

mult_reg1 <- lm(wages_owed ~ claimants + year + usdol, data = wage_clean1)
summary(mult_reg1)

#this one showed us that US DoL filing doesn't really mean much


Call:
lm(formula = wages_owed ~ claimants + year + usdol, data = wage_clean1)

Residuals:
    Min      1Q  Median      3Q     Max 
-185410  -13934  -10392   -6340 5747766 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept) 6717623.8  6004946.7   1.119    0.263    
claimants       863.8      118.8   7.271 4.93e-13 ***
year          -3317.3     2970.8  -1.117    0.264    
usdol         15187.2    31454.6   0.483    0.629    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 146800 on 2209 degrees of freedom
Multiple R-squared:  0.02361,	Adjusted R-squared:  0.02228 
F-statistic:  17.8 on 3 and 2209 DF,  p-value: 2.037e-11



In [7]:
%%R

library(dplyr)
library(readr)

wage_single <- wage %>%
  mutate(
    wages_owed = parse_number(`Wages Owed`),
    claimants = as.numeric(`Number of Claimants on Case`)
  ) %>%
  filter(
    !is.na(wages_owed),
    !is.na(claimants),
    claimants > 0
  )

single_reg <- lm(wages_owed ~ claimants, data = wage_single)
summary(single_reg)


Call:
lm(formula = wages_owed ~ claimants, data = wage_single)

Residuals:
    Min      1Q  Median      3Q     Max 
-180321  -12730  -11734   -7949 5748793 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  12341.2     3231.3   3.819 0.000138 ***
claimants      832.3      115.5   7.207 7.82e-13 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 146800 on 2211 degrees of freedom
Multiple R-squared:  0.02295,	Adjusted R-squared:  0.02251 
F-statistic: 51.94 on 1 and 2211 DF,  p-value: 7.817e-13



In [8]:
%%R

library(dplyr)
library(readr)

wage_clean2 <- wage %>%
  mutate(
    wages_owed = parse_number(`Wages Owed`),
    claimants = as.numeric(`Number of Claimants on Case`),
    wages_per_claimant = wages_owed / claimants,
    usdol = as.integer(toupper(`USDOL?`) == "YES"),
    year = as.numeric(`Year of Case`),
    city_clean = str_trim(City),
    city = as.factor(City)
  ) %>%
  filter(
    !is.na(wages_owed),
    !is.na(claimants),
    claimants > 0,
    wages_per_claimant > 0
  )

mult_per_claimant <- lm(log(wages_per_claimant) ~ year + usdol + city, data = wage_clean2)
summary(mult_per_claimant)


Call:
lm(formula = log(wages_per_claimant) ~ year + usdol + city, data = wage_clean2)

Residuals:
    Min      1Q  Median      3Q     Max 
-7.1526 -1.0433 -0.0712  0.9967  5.8407 

Coefficients:
                      Estimate Std. Error t value Pr(>|t|)    
(Intercept)          655.29283   71.36069   9.183   <2e-16 ***
year                  -0.32149    0.03531  -9.104   <2e-16 ***
usdol                  0.37601    0.39776   0.945   0.3446    
cityAstoria            1.21573    1.27727   0.952   0.3413    
cityBayside            2.56092    1.39263   1.839   0.0661 .  
cityBriarwood          1.66518    1.60788   1.036   0.3005    
cityBroad Channel      1.47543    2.15783   0.684   0.4942    
cityBronx              1.91077    1.25138   1.527   0.1269    
cityBrooklyn           1.91263    1.24786   1.533   0.1255    
cityCollege Point      1.31614    1.41226   0.932   0.3515    
cityCorona             2.12895    1.29861   1.639   0.1013    
cityDouglaston         2.25280    1.76212   1.27

In [9]:
%%R

require("DescTools")

Loading required package: DescTools


In [10]:
%%R

library(dplyr)
library(readr)

wage_clean3 <- wage %>%
  mutate(
    wages_owed = parse_number(`Wages Owed`),
    claimants = as.numeric(`Number of Claimants on Case`),
    wages_per_claimant = wages_owed / claimants,
    usdol = as.integer(toupper(`USDOL?`) == "YES"),
    year = as.numeric(`Year of Case`),
    city_clean = str_trim(City),
    city = as.factor(City)
  ) %>%
  filter(
    !is.na(wages_owed),
    !is.na(claimants),
    claimants > 0,
    wages_per_claimant > 0
  )

mult_dol <- glm(usdol ~ claimants + wages_owed + year, data = wage_clean3, family = binomial)
summary(mult_dol)

#PseudoR2(mult_dol)


Call:
glm(formula = usdol ~ claimants + wages_owed + year, family = binomial, 
    data = wage_clean3)

Coefficients:
              Estimate Std. Error z value Pr(>|z|)
(Intercept)  1.709e+01  4.131e+02   0.041    0.967
claimants   -4.804e-03  1.147e-02  -0.419    0.675
wages_owed   3.866e-07  7.510e-07   0.515    0.607
year        -1.072e-02  2.044e-01  -0.052    0.958

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 246.67  on 2212  degrees of freedom
Residual deviance: 246.32  on 2209  degrees of freedom
AIC: 254.32

Number of Fisher Scoring iterations: 7



In [18]:
%%R

library(dplyr)
library(readr)

wages_by_year <- wage %>%
  mutate(
    wages_clean = parse_number(`Wages Owed`)
  ) %>%
  group_by(`Year of Case`) %>%
  summarise(
    total_wages_stolen = sum(wages_clean, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(`Year of Case`)

wages_by_year

# A tibble: 5 × 2
  `Year of Case` total_wages_stolen
           <dbl>              <dbl>
1           2020           16008665
2           2021           11041838
3           2022           33877345
4           2023           12205190
5           2024            5122436
